# Final Model

## Objective

This notebook trains the final pathogenicity prediction model using the most informative genomic and clinical features identified during baseline experimentation.

### Features

- Chromosome
- GeneSymbol
- mutation_type
- OriginSimple
- ReviewStatus
- ref_length
- alt_length
- length_difference

### Model

Random Forest Classifier with:

- OneHotEncoder
- Class balancing
- Increased number of trees

The goal is to obtain the best predictive performance while maintaining interpretability.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

In [ ]:
df = pd.read_csv(

    "../data/processed/clinvar_features.csv"

)

df.shape

In [ ]:
features = [
    "Chromosome",
    "GeneSymbol",
    "mutation_type",
    "OriginSimple",
    "ReviewStatus",
    "ref_length",
    "alt_length",
    "length_difference"
]

target = "target"

In [ ]:
df_final = df[
    features + [target]
].dropna().copy()

df_final.shape

In [ ]:
categorical_features = [
    "Chromosome",
    "GeneSymbol",
    "mutation_type",
    "OriginSimple",
    "ReviewStatus"
]

numeric_features = [
    "ref_length",
    "alt_length",
    "length_difference"
]

for col in categorical_features:
    df_final[col] = df_final[col].astype(str)

In [ ]:
sample_final = df_final.sample(
    n=50000,
    random_state=42
).copy()

sample_final.shape

In [ ]:
X = sample_final[features]

y = sample_final[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

In [ ]:
final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        ("classifier",
         RandomForestClassifier(
             n_estimators=300,
             max_depth=20,
             class_weight="balanced",
             random_state=42,
             n_jobs=-1
         ))
    ]
)

In [ ]:
final_model.fit(
    X_train,
    y_train
)

In [ ]:
predictions = final_model.predict(
    X_test
)

In [ ]:
print(
    classification_report(
        y_test,
        predictions
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    predictions
)

print(cm)

In [ ]:
rf = final_model.named_steps["classifier"]

importances = rf.feature_importances_

In [ ]:
feature_names = (
    final_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

importance_df.head(20)

In [ ]:
import matplotlib.pyplot as plt

top20 = importance_df.head(20)

plt.figure(figsize=(10,6))

plt.barh(
    top20["feature"],
    top20["importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Most Important Features")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("../reports", exist_ok=True)

top20 = importance_df.head(20)

plt.figure(figsize=(10, 8))

plt.barh(
    top20["feature"],
    top20["importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Most Important Features")

plt.gca().invert_yaxis()
plt.tight_layout()

plt.savefig(
    "../reports/feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    final_model,
    "../models/final_model.pkl"
)

In [ ]:
importance_df.to_csv(
    "../reports/feature_importance.csv",
    index=False
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.tight_layout()

plt.savefig(
    "../reports/confusion_matrix.png"
)

plt.show()